Get OpenAI client

In [1]:
from openai import OpenAI
openai_client = OpenAI()

Ask standard GPT LLM a question 

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Yes, likely — but it depends on the course’s enrollment rules and whether it’s still open.

If you want, send me:
- the course name
- the platform or school
- whether you want to join as a student, auditor, or late enrollee

and I can help you figure out the exact next step or draft a message to the instructor/admin.


Download the FAQ data through load_faq_data function

In [7]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

Define a search function which encapsultes the

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search('how do i run ollama ?')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

Tell the GPT model about the actual search function

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

Tell the LLM about the search function so that it considers it as a tool it can use

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

In [15]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join new student enrollment registration"}', call_id='call_yHTzBfWGAcDAhsLtPDxoAWfE', name='search', type='function_call', id='fc_0b0c0be1a9384441006ad8b62a879887d292ccf88b3081bc20', async_=None, caller=None, namespace=None, status='completed')

In [20]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered late can I join new student enrollment registration'}

In [24]:
results = search(**args)
results_json = json.dumps(results, indent=2)
print(results_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "1e829a8c6f",
    "course": "llm-zoomcamp",
    "section": "Module 2: Vector Search",
    "question": "Do I need a new GitHub repo for Module 2, or just a new codespace?",
    "answer": "Just a new codespace. A codespace is an environment (see *Can I run the course locally instead of Codespaces?*); you create it from your existing repository, so you don't need a new GitHub repo.\n\nUse a separate codespace for Module 2 because the vector-search dependencies are fairly heavy. Keeping them isolated means you can simply stop or delete that codespace when you're done, rather than leaving the extra weight in your Module 1 environment. Setup is quick.\n\nSince

These results are sent back to the LLM

In [26]:
messages.extend(response.output)

# Append the function call output to the messages list to provide a full history to the LLM
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
})

In [34]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join new student enrollment registration"}', call_id='call_yHTzBfWGAcDAhsLtPDxoAWfE', name='search', type='function_call', id='fc_0b0c0be1a9384441006ad8b62a879887d292ccf88b3081bc20', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_yHTzBfWGAcDAhsLtPDxoAWfE',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "1e829a8c6f",\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Vector Search",\n    "question": "Do I need a new GitHub repo for Module 2, or jus

In [36]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes, you can still join and start learning.

If you want a certificate, you’ll need to submit your project while the course is still accepting submissions.


## Global agent steps 
1. Make a call to the LLM (let him make the choice to use the search function or not) <-- first call to the LLM
2. LLM decides to invoke search('params')
3. We invoke the search, we have the results 
4. Send the results back to the LLM <-- second call to the LLM
5. LLM procesees the results 
6. LLM gives the answer